# Conversion GGUF (Bonsai 27B) -> SafeTensors Ternaire AI Box

Ce notebook convertit un modèle `.gguf` (ex. Bonsai 27B `q1_0`) en format **AI Box** (`model.safetensors` + `quant_config.json` + `config.json` + `tokenizer.json`) avec rotation **Hadamard `H_128`** et packing **2-bit** (`00 -> 0`, `01 -> +1`, `11 -> -1`), **sans aucun entraînement**.

In [ ]:
# CELLULE 1 : Cloner le dépôt ai-box & installer les dépendances
!git clone https://github.com/thienban/ai-box.git || true
%cd /content/ai-box
!git pull
!pip install -q gguf safetensors huggingface_hub torch numpy

In [ ]:
# CELLULE 2 : Télécharger le .gguf depuis Hugging Face (détection automatique du fichier)
from huggingface_hub import hf_hub_download, list_repo_files

HF_REPO_ID = "prism-ml/Bonsai-27B-gguf"  # Adaptez selon le dépôt GGUF choisi
gguf_files = [f for f in list_repo_files(HF_REPO_ID) if f.endswith(".gguf")]
print("Fichiers GGUF trouvés :", gguf_files)

target_file = next((f for f in gguf_files if "Q1_0" in f or "q1_0" in f), gguf_files[0])
GGUF_PATH = hf_hub_download(repo_id=HF_REPO_ID, filename=target_file)
print("Fichier GGUF prêt :", GGUF_PATH)

In [ ]:
# CELLULE 3 : Conversion GGUF -> SafeTensors AI Box
# Mettre --max-layers 2 pour un test rapide (~10s), ou retirer --max-layers pour les 64 couches complètes.
!python scripts/convert_gguf_to_aibox.py \
    --gguf "{GGUF_PATH}" \
    --out /content/bonsai-aibox \
    --max-layers 2

!ls -lh /content/bonsai-aibox

In [ ]:
# CELLULE 4 : Vérifier l'export SafeTensors en Python (sans installer Rust sur Colab)
import json
from safetensors import safe_open

with open("/content/bonsai-aibox/config.json") as f:
    print("config.json :", json.load(f))
with open("/content/bonsai-aibox/quant_config.json") as f:
    print("quant_config.json :", json.load(f))

with safe_open("/content/bonsai-aibox/model.safetensors", framework="pt") as st:
    keys = list(st.keys())
    print(f"Total tenseurs : {len(keys)}")
    for k in keys[:6]:
        t = st.get_tensor(k)
        print(f"  - {k}: shape={tuple(t.shape)}, dtype={t.dtype}")

In [ ]:
# CELLULE 5 (Optionnel) : Pousser le modèle converti vers votre Hugging Face (pour `aibox pull` sur PC)
# from huggingface_hub import HfApi
# api = HfApi(token="hf_VOTRE_TOKEN")
# api.create_repo(repo_id="votre-user/bonsai-27b-aibox", exist_ok=True, private=True)
# api.upload_folder(folder_path="/content/bonsai-aibox", repo_id="votre-user/bonsai-27b-aibox")